# Lab M.4 &mdash; Langfuse MCP in a ReAct Agent

**Day 3 &middot; MCP &middot; about 20 min &middot; in your sandbox**

### What you'll do
- Connect to the Langfuse MCP server from Python and list its tools
- Wrap the tools you want as LangChain tools
- Give them to a ReAct agent and watch it reason, call tools and answer

> **How this lab works.** Run each cell in order with **Shift + Enter** and read what it prints. There is nothing to fill in and nothing is graded.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "aicp-lab-m-4")
os.makedirs(WORK, exist_ok=True)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If something it needs is missing, print a short note instead of crashing."""
    try:
        return fn()
    except NameError as exc:
        print(f"(an earlier cell has not run yet: {exc} -- run the cells above, then this one)")
        return default

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # ask your trainer")
        print("  export LAB_LLM_MODEL=...       # ask your trainer")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- tell your trainer)")

## Concept

A ReAct agent loops: **reason &rarr; act &rarr; observe**, until it can answer. `act` means calling
a tool, and LangChain does not care where that tool came from.

So putting an MCP server inside a ReAct agent is one translation step:

```
MCP tools/list  ->  StructuredTool  ->  create_agent(tools=[...])
```

`name`, `description` and `inputSchema` come straight across &mdash; `StructuredTool` accepts JSON
Schema as `args_schema`, so there is no conversion to write.

## Step 1 &mdash; Talk to the server

Two calls: `initialize`, then `tools/list`. Everything the sandbox needs is already in your
environment.

One thing to know before the agent starts calling tools: Langfuse Cloud allows **30 API calls a
minute for the whole project**, and every participant on this course shares that project. An agent
run makes about ten calls, so `429 Too Many Requests` is a normal thing to meet here. The
client below waits out the `Retry-After` the server sends rather than failing the run &mdash; a
shared credential has a shared budget, and code that talks to one has to expect it.

In [ ]:
import base64, urllib.request, urllib.error

LF_URL  = os.environ.get("LANGFUSE_HOST", "").rstrip("/") + "/api/public/mcp"
LF_AUTH = base64.b64encode(f"{os.environ.get('LANGFUSE_PUBLIC_KEY','')}:"
                           f"{os.environ.get('LANGFUSE_SECRET_KEY','')}".encode()).decode()
_sid = {"v": None}


def langfuse_ready() -> bool:
    return all(os.environ.get(k) for k in
               ("LANGFUSE_HOST", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"))


def mcp(method: str, params: dict = None, tries: int = 4) -> dict:
    """One JSON-RPC call to the MCP server. Returns the whole envelope -- never raises.

    Langfuse Cloud allows 30 API calls a minute for the whole project, and everyone on this
    course shares it, so a 429 here is a busy neighbour rather than a bug. Wait out the
    Retry-After it sends and try again; if it is still busy, hand the failure back as an
    error envelope so the agent can read it instead of dying mid-run.
    """
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}).encode()
    for attempt in range(1, tries + 1):
        req = urllib.request.Request(LF_URL, data=body, method="POST")
        req.add_header("Content-Type", "application/json")
        req.add_header("Accept", "application/json, text/event-stream")
        req.add_header("Authorization", "Basic " + LF_AUTH)
        if _sid["v"]:
            req.add_header("Mcp-Session-Id", _sid["v"])
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                raw, sid = r.read().decode(), r.headers.get("mcp-session-id")
            break
        except urllib.error.HTTPError as exc:
            if exc.code == 429 and attempt < tries:
                wait = int(exc.headers.get("Retry-After") or 20)
                print(f"  (Langfuse is rate limited -- waiting {wait}s, attempt {attempt}/{tries})")
                time.sleep(wait)
                continue
            return {"error": {"code": exc.code, "message": f"HTTP {exc.code} {exc.reason}"}}
        except Exception as exc:
            return {"error": {"code": -1, "message": f"{type(exc).__name__}: {exc}"}}
    if sid:
        _sid["v"] = sid
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw)


if langfuse_ready():
    mcp("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                       "clientInfo": {"name": "lab-m-4", "version": "1.0"}})
    published = guard(lambda: mcp("tools/list", {}).get("result", {}).get("tools", []), [])
    print(f"the server offers {len(published)} tools, for example:")
    for t in published[:5]:
        print(f"  {t['name']:24} {t.get('description','')[:56]}")
    if not published:
        print("  (none came back -- if you were rate limited, wait a minute and re-run this cell)")
else:
    published = []
    print("Langfuse is not configured here. Set LANGFUSE_HOST / _PUBLIC_KEY / _SECRET_KEY.")

## Step 2 &mdash; Wrap the ones you want

You are not binding all of them. Pick the few that answer the question you care about &mdash; every
tool you hand the agent is one more thing it can choose wrongly, and one more schema it carries on
every turn.

Look at `description=` in the next cell. That field of the MCP spec is the sentence the model reads
when it decides whether to call a tool, so it is copied across unchanged.

In [ ]:
from langchain_core.tools import StructuredTool

WANTED = ["getMetricsSchema",            # which measures queryMetrics accepts
          "queryMetrics",               # aggregate: how many, how slow, by name
          "getObservationFieldSchema",  # which field names listObservations accepts
          "getObservationFilterSchema", # which filters listObservations accepts
          "listObservations",           # find individual ones
          "getObservation"]             # fetch one in full


def as_langchain_tool(spec: dict) -> StructuredTool:
    """One MCP tool definition -> one LangChain tool."""
    def call(**kwargs):
        env = mcp("tools/call", {"name": spec["name"], "arguments": kwargs})
        if "error" in env:                       # let the model see failures, or it will loop
            return "ERROR: " + str(env["error"].get("message", env["error"]))[:400]
        parts = env.get("result", {}).get("content", [])
        return "".join(p.get("text", "") for p in parts)[:1500]

    return StructuredTool.from_function(
        func=call,
        name=spec["name"],
        description=spec["description"][:900],
        args_schema=spec.get("inputSchema") or {"type": "object", "properties": {}},
    )


tools = [as_langchain_tool(s) for s in published if s["name"] in WANTED]
print("bridged:", [t.name for t in tools])

## Step 3 &mdash; Give them to a ReAct agent

`create_agent` builds the reason&ndash;act&ndash;observe loop. It has no idea these tools speak MCP.

In [ ]:
import datetime
from langchain.agents import create_agent
from langgraph.errors import GraphRecursionError

NOW = datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

SYSTEM = (
    "You answer questions about a Langfuse project using the tools provided.\n"
    "Call getMetricsSchema before any queryMetrics call, and getObservationFieldSchema "
    "before any listObservations call, and getObservationFilterSchema before any filter. Use only the names they list -- guessing a field "
    "or measure name costs several wasted turns.\n"
    f"The time now is {NOW} (UTC). Write timestamps in ISO 8601, like {NOW}.\n"
    "Add no filters unless the question asks for one. Latency is measured in milliseconds.\n"
    "If a tool returns an ERROR, read it and correct your arguments. Be brief."
)


def ask(question: str) -> str:
    agent = create_agent(model=get_llm(), tools=tools, system_prompt=SYSTEM)
    try:
        out = agent.invoke({"messages": [("user", question)]}, config={"recursion_limit": 25})
    except GraphRecursionError:
        return "(stopped after 25 steps without an answer -- ask a narrower question)"
    for m in out["messages"]:                       # show the act/observe steps
        for tc in (getattr(m, "tool_calls", None) or []):
            print("  act:", tc["name"], json.dumps(tc["args"])[:88])
    return out["messages"][-1].content


QUESTION = ("Find the slowest observation in the last 30 days, fetch it in full, "
            "and explain in three sentences what it was doing.")

if llm_ready() and tools:
    print(guard(lambda: ask(QUESTION)))
else:
    print("Run it for real needs the model and Langfuse. See the setup cell.")

## What you just saw

Read the `act:` lines. That one English sentence became a chain: ask the schema what exists,
`queryMetrics` to rank observations by latency, `listObservations` to identify the slow one,
`getObservation` to read it in full &mdash; then the model wrote the summary. **You sequenced none
of that.** You handed it six tools and a question in English.

On this model and this question, expect **five to ten `act:` lines and under a
minute**, with visible retries: the model proposes an argument, the server returns `ERROR:`, it
reads it and tries again. That recovery is why the wrapper *returns* error text instead of raising
&mdash; a raise ends the turn, a returned string lets the model correct itself. The agent is
slower and more argumentative than the four lines of summary suggest. Watch the middle, not just
the answer. Then check the answer in the Langfuse UI: in one test run the model named a
0 ms step as the slowest.

`getObservationFieldSchema` is in `WANTED` for a reason worth knowing. An earlier version bound
only the other four, and the model tried to call `getObservationFieldSchema` anyway &mdash; a tool
it had never been given &mdash; then spent six turns guessing `start_time` versus `startTime`.
**It named the tool it was missing.** What an agent reaches for and cannot find is the best signal
you get about what to bind next.

And the honest result of binding it: the field-name guessing stopped, the answer got richer &mdash;
and the **turn count did not drop at all**. The model simply moved its uncertainty to the shape of
`queryMetrics`. Adding a tool removed one failure mode; it did not buy speed. Measure that before
you promise it to anyone.

That is the whole integration: **twenty lines, and an MCP server is just tools now.**

## Now interrogate it yourself

`ask()` is the entire interface. Edit `QUESTION` and re-run the cell, or call it directly:

```python
ask("Which observation names appear most often in the last 30 days?")
ask("Give me average and maximum latency by observation name for the last 30 days.")
ask("List the three most recent observations with their name, type and latency.")
ask("Find the most recent GENERATION and tell me which model it used.")
ask("Are there any observations with level ERROR? Show me the most recent one in full.")
```

Questions answered by one aggregate finish in two or three steps. Ones that end in "&hellip;and
fetch it in full" chain three tools, because finding a thing and reading a thing are different
tools &mdash; the agent works that out from the descriptions, not from you.

## Your turn

- Add `listPrompts` to `WANTED` and ask something that needs it. One line, one more capability.
- Truncate the description to 30 characters and re-run. Same tools, worse choices &mdash; the prose
  was doing the work.
- Point the same wrapper at the Jira server from Lab M.1. Only `mcp()` knows it talks to Langfuse:
  set `LF_URL` to `os.environ["JIRA_MCP_URL"]` and `LF_AUTH` to `os.environ["JIRA_MCP_AUTH"]`.